This code is ported from the Maple program as seen in Hairer & Lubich paper (Asymptotic expansions and backward analysis for numerical integrators)

In [ ]:
import sympy as sp

h = sp.symbols('h')

def modified_equation(method, f, y, N):
    phi = method(f, y, h)
    fcoe = {1: f(y)}
    for n in range(2, N + 1):
        modeq = sum((h**j*fcoe[j+1] for j in range(n - 1)), sp.zeros(*y.shape))
        diffy, y_tilde = y, y
        for k in range(1, n + 1):
            diffy = diffy.jacobian(y) * modeq
            y_tilde = y_tilde + h**k * diffy / sp.factorial(k)
        res = y_tilde - phi
        fcoe[n] = -(res.diff(h, n).subs(h, 0) / sp.factorial(n)).applyfunc(sp.simplify)
    return fcoe        

def explicit_euler(f, y, h):
    return y + h*f(y)

def heun(f, y, h):
    return y + h/2*(f(y) + f(y + h*f(y)))

def symplectic_euler(f, y, h):
    q, p = y
    p1 = p + h*f(y)[1]
    q1 = q + h*f(sp.Matrix([q, p1]))[0]
    return sp.Matrix([q1, p1])

def stormer_verlet(f, y, h):
    q, p = y
    p_half = p + h/2*f(y)[1]
    q1 = q + h*f(sp.Matrix([q, p_half]))[0]
    p1 = p_half + h/2*f(sp.Matrix([q1, p_half]))[1]
    return sp.Matrix([q1, p1])


In [15]:
y = sp.symbols('y')
Y = sp.Matrix([y])
square = lambda Y: Y.applyfunc(lambda v: v**2)

q, p = sp.symbols('q p')
QP = sp.Matrix([q, p])
pendulum = lambda Y: sp.Matrix([Y[1], -sp.sin(Y[0])])

ee = modified_equation(explicit_euler, square, Y, 6)
he = modified_equation(heun, square, Y, 4)
se = modified_equation(symplectic_euler, pendulum, QP, 4)
sv = modified_equation(stormer_verlet, pendulum, QP, 4)

print(ee)
print(he)
print(se)
print(sv)


{1: Matrix([[y**2]]), 2: Matrix([[-y**3]]), 3: Matrix([[3*y**4/2]]), 4: Matrix([[-8*y**5/3]]), 5: Matrix([[31*y**6/6]]), 6: Matrix([[-157*y**7/15]])}
{1: Matrix([[y**2]]), 2: Matrix([[0]]), 3: Matrix([[-y**4/2]]), 4: Matrix([[y**5/2]])}
{1: Matrix([
[      p],
[-sin(q)]]), 2: Matrix([
[ -sin(q)/2],
[p*cos(q)/2]]), 3: Matrix([
[                  p*cos(q)/6],
[p**2*sin(q)/12 - sin(2*q)/12]]), 4: Matrix([
[ -sin(2*q)/24],
[p*cos(2*q)/12]])}
{1: Matrix([
[      p],
[-sin(q)]]), 2: Matrix([
[0],
[0]]), 3: Matrix([
[               p*cos(q)/6],
[(p**2 + cos(q))*sin(q)/12]]), 4: Matrix([
[0],
[0]])}
